<a href="https://colab.research.google.com/github/alexcho2823-del/ds2002-fa26/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
SELECT t.track_id, t.title, t.genre, t.seconds, a.name AS artist_name, a.country
FROM tracks t
JOIN artists a ON t.artist_id = a.artist_id;
''')

,track_id,title,genre,seconds,artist_name,country
0,10,Skyline,Pop,201,Nova Waves,US
1,11,Undertow,Pop,240,Nova Waves,US
2,12,Foothills,Folk,185,The Blue Ridge,US
3,13,Aurora,Electronic,300,Kestrel,UK
4,14,Nightfall,Electronic,275,Kestrel,UK
5,15,Sol,Latin,210,Marisol,ES
6,16,Coastline,Folk,199,The Blue Ridge,US
7,17,Ridgeline,Folk,225,The Blue Ridge,US
8,18,Untitled Demo,None,150,Kestrel,UK


This is the straight inner join on artist id, and since every track already has a non null artist id, I think it is safe to use JOIN instead of LEFT JOIN and still get all the intended rows

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT genre, AVG(seconds) AS avg_seconds
FROM tracks
GROUP BY genre
ORDER BY avg_seconds DESC
LIMIT 1;
''')

,genre,avg_seconds
0,Electronic,287.5


Grouping by the genre and ordering by the computed average is what makes the average show up in the result.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
SELECT user, COUNT(*) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user;
''')

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


COUNT will counts every row while COUNT collapses the repeats, so the two columns side by side will show you a repeat-listener from someone sampling different tracks.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.track_id IS NULL;
''')

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


The LEFT JOIN will keep every track even when there is no matching play, and filtering the p.track id IS NULL isolates exactly the tracks where that match never happened, giving the expected 2 rows.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT a.name AS artist_name,
       SUM(t.seconds) AS total_seconds,
       ROUND(SUM(t.seconds) / 60.0, 1) AS total_minutes
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.artist_id
ORDER BY total_seconds DESC;
''')

,artist_name,total_seconds,total_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


Starting from plays and joining outward to tracks and artists means each play contributes its track's seconds once per play. Therefore an artist with fewer tracks but more repeat plays can still rank higher.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL;
-- WHERE genre != 'Pop' would NOT have caught this row: SQL compares NULL to
-- anything as NULL (not true or false), so a NULL genre fails both
-- "= 'Pop'" and "!= 'Pop'" and silently drops out of the result either way.
''')

,track_id,title
0,18,Untitled Demo


IS NULL is the only way to actually test for missingness in SQL, which the code comment calls out as the reason != would have quietly hidden this row instead of catching it.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT played_on,
       COUNT(*) AS plays,
       COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on;
''')

,played_on,plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


Grouping the played_on text column tallies both totals per date, and because the dates are stored as 'YYYY-MM-DD' strings, a plain alphabetical ORDER BY also happens to be chronological order.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
q1 = q('''SELECT t.track_id, t.title, t.genre, t.seconds, a.name AS artist_name, a.country
FROM tracks t JOIN artists a ON t.artist_id = a.artist_id''')
q3 = q('''SELECT user, COUNT(*) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays GROUP BY user''')
q4 = q('''SELECT t.track_id, t.title FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id WHERE p.track_id IS NULL''')

assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

After completing this Challenge Set, the one that gave me the most trouble was Q4. It was because I wrote it with a plain JOIN between the plays and tracks, and then I filtered wher WHERE IS NULL. That returend zero rows instead of the intended 2, because the inner join only keeps where the tables and rows both have a match. It drops the tracks I was trying to find so there was nothing left for IS NULL. I had the right idea for finding the missing data but I paired it with the wrong join type. After switchign it to the left JOIN kept all the tracks and everyting started to work.


